# Apprendre à partir des données : régression, descente de gradient, k-moyennes

**Cours** : Intelligence artificielle · Master 2 (ANG, LGC, DID) · Université Yahia Farès, Médéa
**Chapitre 3** : Types d'IA et approches · Quelques algorithmes d'apprentissage automatique
**Durée** : environ 1 h 30

### Ce que vous allez faire
- Refaire en Python les calculs de l'exercice 2 a du TD du chapitre 3 : la droite de régression par la méthode des moindres carrés, puis la vérifier avec NumPy et scikit-learn.
- Suivre pas à pas la descente de gradient de l'exercice 2 b, voir le rôle du taux d'apprentissage, puis l'utiliser pour faire « apprendre » la droite de régression à la machine.
- Dérouler les k-moyennes de l'exercice 2 d à la main, puis comparer avec `KMeans` de scikit-learn et la méthode du coude.

### Mode d'emploi
Exécutez chaque cellule de haut en bas : cliquez sur le bouton ▶ à sa gauche, ou appuyez sur Maj + Entrée. **Vous n'avez jamais à écrire de code.** Modifiez seulement les champs des cellules intitulées « À MODIFIER ».

- Dans Google Colab, le menu « Exécution → Tout exécuter » (*Runtime → Run all*) lance tout le carnet ; les champs « À MODIFIER » s'affichent comme des curseurs ou des zones de texte.
- Dans Jupyter, ces champs sont de simples commentaires `#@param` : changez seulement la valeur placée après le signe `=`.
- Après avoir modifié une valeur, relancez la cellule, puis les cellules qui la suivent.

Gardez à portée de main votre corrigé du TD et une calculatrice : avant chaque cellule, **prédisez** le résultat ; après, comparez avec la rubrique « Ce que vous devez voir ». Toutes les données de ce carnet sont **fictives**, minuscules, pour que les calculs se refassent à la main.

## 1. La régression linéaire (TD 3, exercice 2 a)
La régression linéaire est de l'**apprentissage supervisé** : on connaît, pour chaque exemple, un attribut `x` et la bonne réponse `y`, et l'on cherche la droite ŷ = a·x + b qui passe au plus près des points. On a relevé, pour cinq étudiants (données fictives), le nombre d'heures `x` consacrées chaque semaine à la lecture et le nombre `y` de livres lus pendant le semestre : (1 ; 2), (2 ; 3), (3 ; 5), (4 ; 4), (5 ; 6).

La méthode des moindres carrés donne :

a = Σ(x − x̄)(y − ȳ) / Σ(x − x̄)²  et  b = ȳ − a·x̄

La cellule calcule les moyennes x̄ et ȳ, puis le tableau des écarts du corrigé, colonne par colonne. Avec `pandas`, un calcul écrit sur une colonne s'applique à toutes ses lignes. Le curseur `livres_etudiant_5` permet de changer la valeur du cinquième étudiant (6 dans le TD).

**Avant d'exécuter :** que valent x̄ et ȳ ? Quelle est la somme de la colonne (x − x̄)(y − ȳ) ?

In [ ]:
#@title À MODIFIER : les données de l'exercice 2 a et le tableau des écarts
livres_etudiant_5 = 6  #@param {type:"slider", min:0, max:15, step:1}

import pandas as pd
x = [1, 2, 3, 4, 5]                        # heures de lecture par semaine (fictif)
y = [2, 3, 5, 4, livres_etudiant_5]        # livres lus pendant le semestre (fictif)
x_moyen = sum(x) / len(x)
y_moyen = sum(y) / len(y)
print("x̄ =", x_moyen, "  ȳ =", y_moyen)
tableau = pd.DataFrame({"x": x, "y": y})
tableau["x - x̄"] = tableau["x"] - x_moyen                  # écart à la moyenne, ligne par ligne
tableau["y - ȳ"] = tableau["y"] - y_moyen
tableau["(x - x̄)(y - ȳ)"] = tableau["x - x̄"] * tableau["y - ȳ"]
tableau["(x - x̄)²"] = tableau["x - x̄"] ** 2
tableau

### Ce que vous devez voir
`x̄ = 3.0   ȳ = 4.0`, puis le tableau du corrigé :

VOIR_TABLEAU_ECARTS

Python écrit les nombres décimaux avec un point : `-2.0` se lit −2,0.

**Observez :** faites la somme de chaque colonne de tête. Les colonnes x − x̄ et y − ȳ ont toujours une somme nulle : pourquoi ?

### Calculer a et b
`sum()` additionne une colonne. On applique ensuite les deux formules du cours.

**Avant d'exécuter :** avec votre corrigé, que valent a et b ?

In [ ]:
#@title Calculer a et b par la méthode des moindres carrés
somme_produits = tableau["(x - x̄)(y - ȳ)"].sum()
somme_carres = tableau["(x - x̄)²"].sum()
a = somme_produits / somme_carres
b = y_moyen - a * x_moyen
print("Σ(x - x̄)(y - ȳ) =", somme_produits, "   Σ(x - x̄)² =", somme_carres)
print(f"a = {a:.2f}   b = {b:.2f}")
print(f"Droite de régression : ŷ = {a:.2f} x + {b:.2f}")

### Ce que vous devez voir
```
VOIR_AB
```
C'est le corrigé : a = 9 / 10 = 0,9 et b = 4 − 0,9 × 3 = 1,3. Dans ces données fictives, chaque heure de lecture hebdomadaire supplémentaire est associée à 0,9 livre de plus sur le semestre.

**Observez :** « associée à » n'est pas « cause ». Lire plus d'heures fait-il lire plus de livres, ou bien les étudiants qui aiment lire font-ils les deux ?

### Prédire, puis vérifier les écarts
Le modèle sert à **prédire** : pour `x` heures, ŷ = a·x + b. La cellule calcule aussi les valeurs prédites ŷ pour les cinq étudiants et les écarts y − ŷ (les **résidus**), comme dans la vérification du corrigé.

**Avant d'exécuter :** combien de livres le modèle prédit-il pour 6 heures par semaine ? Que vaut la somme des écarts ?

In [ ]:
#@title À MODIFIER : prédire pour un nombre d'heures, puis vérifier les écarts
heures = 6  #@param {type:"slider", min:0, max:12, step:1}

print(f"Prédiction pour {heures} h par semaine : ŷ = {a * heures + b:.1f} livres")
tableau["ŷ"] = a * tableau["x"] + b                 # les valeurs prédites
tableau["y - ŷ"] = tableau["y"] - tableau["ŷ"]      # les écarts (résidus)
print("Somme des écarts :", round(tableau["y - ŷ"].sum(), 2))
print("Somme des carrés des écarts :", round((tableau["y - ŷ"] ** 2).sum(), 2))
tableau[["x", "y", "ŷ", "y - ŷ"]].round(2)

### Ce que vous devez voir
```
VOIR_PREDICTION
```
VOIR_TABLEAU_RESIDUS

Les valeurs prédites sont 2,2 ; 3,1 ; 4,0 ; 4,9 ; 5,8. La somme des écarts est nulle, comme toujours avec les moindres carrés (Python peut afficher `-0.0`), et la somme de leurs carrés vaut 1,9 : aucune autre droite ne fait moins.

**Modifiez :** placez le curseur `heures` sur 12, puis sur 0. Le modèle prédit 12,1 livres, puis 1,3 livre pour un étudiant qui ne lit jamais. Pourquoi ces **extrapolations**, hors de l'intervalle observé (1 à 5 heures), sont-elles douteuses ?

### Vérifier avec NumPy et scikit-learn
Les bibliothèques font le même calcul en une ligne. `numpy.polyfit(x, y, 1)` cherche le polynôme de degré 1 (une droite) des moindres carrés. Avec scikit-learn, on crée un modèle `LinearRegression`, on l'entraîne avec `fit`, puis on prédit avec `predict` : c'est le vocabulaire de tout l'apprentissage automatique.

**Avant d'exécuter :** les trois méthodes donneront-elles les mêmes a et b ?

In [ ]:
#@title Vérifier avec NumPy et scikit-learn
import numpy as np
from sklearn.linear_model import LinearRegression

pente, ordonnee = np.polyfit(x, y, 1)           # droite des moindres carrés (degré 1)
print(f"NumPy        : a = {pente:.2f}   b = {ordonnee:.2f}")
modele = LinearRegression()
modele.fit([[valeur] for valeur in x], y)       # entraîner : scikit-learn attend une liste de lignes
print(f"scikit-learn : a = {modele.coef_[0]:.2f}   b = {modele.intercept_:.2f}")
print(f"Prédiction de scikit-learn pour 6 h : {modele.predict([[6]])[0]:.1f} livres")

### Ce que vous devez voir
```
VOIR_VERIF
```
Les trois méthodes concordent. Le calcul « à la main » de la cellule précédente n'est pas une étape inutile : c'est exactement ce que font les bibliothèques.

### Le nuage de points et la droite
`plt.scatter` place les points, `plt.plot` trace la droite ; l'étoile orange marque la prédiction pour le nombre d'heures choisi plus haut.

**Avant d'exécuter :** quel étudiant sera le plus loin de la droite ?

In [ ]:
#@title Dessiner le nuage de points et la droite de régression
import matplotlib.pyplot as plt

abscisses = np.linspace(0, max(6, heures) + 1, 50)     # des valeurs de x pour tracer la droite
plt.figure(figsize=(6, 4))
plt.plot(abscisses, a * abscisses + b, color="#eb6834", linewidth=2, label=f"ŷ = {a:.2f} x + {b:.2f}")
plt.scatter(x, y, color="#2a78d6", s=60, zorder=3, label="Étudiants (données fictives)")
plt.scatter([heures], [a * heures + b], color="#eb6834", marker="*", s=250, zorder=3, label=f"Prédiction pour {heures} h")
plt.xlabel("Heures de lecture par semaine (x)")
plt.ylabel("Livres lus pendant le semestre (y)")
plt.title("Régression linéaire (TD 3, exercice 2 a)")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### Ce que vous devez voir
Cinq points bleus, une droite orange qui monte et une étoile en (6 ; 6,7). Le troisième étudiant (3 ; 5) est le plus loin de la droite : son écart vaut 1,0.

**Modifiez :** tout en haut de la partie 1, placez `livres_etudiant_5` sur 12, puis relancez les cellules de la partie 1 dans l'ordre. Que deviennent a et b ? Un seul étudiant atypique suffit-il à changer la droite ? Que faut-il en conclure pour une enquête sur quelques personnes ?

## 2. La descente de gradient (TD 3, exercice 2 b)
La plupart des modèles ne se calculent pas d'un coup : on cherche leurs paramètres **pas à pas**. Imaginez un randonneur dans le brouillard : il ne voit pas la vallée, mais il sent la pente sous ses pieds et fait chaque fois un pas vers le bas. La vallée est la **fonction d'erreur** J(w) ; la pente est sa dérivée J′(w). À chaque étape :

w ← w − η · J′(w)

η (êta) est le **taux d'apprentissage**, la longueur du pas. Prenons J(w) = (w − 3)², minimale en w = 3, de dérivée J′(w) = 2(w − 3), avec w₀ = 0 et η = 0,1. Tapez les nombres décimaux avec un **point** : `0.1`.

**Avant d'exécuter :** calculez w₁, w₂, J(w₀), J(w₁) et J(w₂). Vers quelle valeur la suite se dirige-t-elle ?

In [ ]:
#@title À MODIFIER : la descente de gradient, pas à pas
eta = 0.1  #@param {type:"slider", min:0.01, max:1.2, step:0.01}
w_depart = 0.0  #@param {type:"number"}
nombre_de_pas = 5  #@param {type:"slider", min:1, max:30, step:1}

w = w_depart
chemin = [w]                                  # on garde chaque valeur de w pour le graphique
print(f"étape 0 : w = {w:.4f}   J(w) = {(w - 3) ** 2:.4f}")
for etape in range(1, nombre_de_pas + 1):
    pente = 2 * (w - 3)                       # la dérivée J'(w) au point w
    w = w - eta * pente                       # un pas dans le sens de la descente
    chemin.append(w)
    print(f"étape {etape} : w = {w:.4f}   J(w) = {(w - 3) ** 2:.4f}   (pente utilisée : {pente:.3f})")

### Ce que vous devez voir
```
VOIR_DESCENTE
```
C'est le corrigé : w₁ = 0,6 et w₂ = 1,08 ; J(w₀) = 9, J(w₁) = 5,76 et J(w₂) = 3,6864. L'erreur diminue à chaque pas, et w se dirige vers 3. À chaque étape, la distance au minimum est multipliée par 1 − 2η = 0,8 : 3 ; 2,4 ; 1,92 ; 1,536…

**Modifiez :** placez `nombre_de_pas` sur 30 : w atteint-il exactement 3 ? Puis essayez `eta` = 1 (w oscille sans fin entre 0 et 6), `eta` = 1.1 (w saute par-dessus le minimum, 6,6 puis −1,32 puis 8,18, et s'en éloigne : l'algorithme **diverge**) et `eta` = 0.01 (w n'atteint que 0,29 après cinq pas : l'apprentissage est très lent).

### Voir les pas sur la courbe d'erreur
La courbe bleue est la fonction d'erreur J(w) ; les points orange sont les valeurs successives de w calculées par la cellule précédente.

**Avant d'exécuter :** les pas seront-ils de plus en plus grands ou de plus en plus petits ? Pourquoi ?

In [ ]:
#@title Dessiner la courbe J(w) et les pas de la descente
valeurs_w = np.linspace(-2, 8, 200)
plt.figure(figsize=(6, 4))
plt.plot(valeurs_w, (valeurs_w - 3) ** 2, color="#2a78d6", linewidth=2, label="J(w) = (w - 3)²")
erreurs = [(v - 3) ** 2 for v in chemin]
plt.plot(chemin, erreurs, color="#eb6834", marker="o", linewidth=1, label=f"Pas de la descente (η = {eta})")
plt.xlabel("Valeur du paramètre w")
plt.ylabel("Erreur J(w)")
plt.title("Descente de gradient (TD 3, exercice 2 b)")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### Ce que vous devez voir
Une parabole bleue dont le creux est en w = 3, et six points orange qui descendent le flanc gauche, de w = 0 vers w ≈ 2. Les pas raccourcissent, car la pente s'adoucit près du fond : le randonneur ralentit de lui-même.

**Modifiez :** avec `eta` = 1.1 dans la cellule précédente, relancez les deux cellules : les points sautent d'un flanc à l'autre en montant.

### Faire « apprendre » la droite de régression
Relions les parties 1 et 2. L'erreur de la droite ŷ = a·x + b est la moyenne des carrés des écarts, J(a, b) = (1/n) · Σ(y − ŷ)². Partons d'une droite plate (a = 0, b = 0) et appliquons la même règle, w ← w − η · pente, à chacun des deux paramètres. Les formules des deux pentes sont écrites dans la cellule : `np.mean` fait la moyenne sur les cinq étudiants. On affiche les pas 1, 10, 100 et le dernier.

**Avant d'exécuter :** après 1 000 pas, quelles valeurs de a et b la machine aura-t-elle trouvées ?

In [ ]:
#@title À MODIFIER : retrouver a et b par descente de gradient
eta_droite = 0.05  #@param {type:"number"}
nombre_de_pas_droite = 1000  #@param {type:"slider", min:10, max:2000, step:10}

X = np.array(x, dtype=float)
Y = np.array(y, dtype=float)
a_appris = 0.0                                   # on part d'une droite plate
b_appris = 0.0
for pas in range(1, nombre_de_pas_droite + 1):
    ecarts = Y - (a_appris * X + b_appris)       # les erreurs de la droite actuelle
    a_appris = a_appris - eta_droite * (-2 * np.mean(ecarts * X))   # pente de J selon a
    b_appris = b_appris - eta_droite * (-2 * np.mean(ecarts))       # pente de J selon b
    if pas in (1, 10, 100) or pas == nombre_de_pas_droite:
        erreur = np.mean((Y - (a_appris * X + b_appris)) ** 2)
        print(f"pas {pas:4d} : a = {a_appris:.4f}   b = {b_appris:.4f}   erreur J = {erreur:.4f}")

### Ce que vous devez voir
```
VOIR_APPRENDRE
```
Pas à pas, la machine retrouve a = 0,9 et b = 1,3 sans connaître les formules des moindres carrés. L'erreur finale, 0,38, est la somme des carrés des écarts (1,9) divisée par les 5 étudiants. **Apprendre**, pour un modèle, c'est exactement cela : ajuster ses paramètres pour diminuer son erreur sur des exemples. Un grand modèle de langage fait de même, avec des milliards de paramètres au lieu de deux.

**Modifiez :** essayez `nombre_de_pas_droite` = 100 (la droite n'est pas encore tout à fait trouvée), puis `eta_droite` = 0.1 : les nombres explosent (`inf`, `nan`, avec un avertissement *overflow*), car le pas est trop grand.

## 3. Les k-moyennes (TD 3, exercice 2 d)
Le regroupement (*clustering*) est de l'**apprentissage non supervisé** : aucune étiquette, on cherche des groupes d'objets semblables. L'algorithme des k-moyennes (*k-means*) alterne deux opérations : **affecter** chaque point au centre le plus proche, puis **recalculer** chaque centre comme la moyenne des points de son groupe.

Six personnages d'une pièce de théâtre ont respectivement 1, 3, 4, 9, 10 et 14 répliques (données fictives). On cherche k = 2 groupes en partant des centres μ₁ = 1 et μ₂ = 4. Sur une seule dimension, la distance entre deux nombres est la valeur absolue de leur différence, `abs()`.

**Avant d'exécuter :** remplissez le tableau de l'itération 1 sur papier. Dans quel groupe va le point 3 ?

In [ ]:
#@title À MODIFIER : les centres de départ, puis l'itération 1 à la main
mu1_depart = 1  #@param {type:"number"}
mu2_depart = 4  #@param {type:"number"}

repliques = [1, 3, 4, 9, 10, 14]            # répliques de six personnages (fictif)
lignes = []
for point in repliques:
    d1 = abs(point - mu1_depart)            # distance au centre 1
    d2 = abs(point - mu2_depart)            # distance au centre 2
    if d1 <= d2:
        groupe = 1                          # plus proche de μ1 (ou à égalité)
    else:
        groupe = 2
    lignes.append([point, d1, d2, groupe])
pd.DataFrame(lignes, columns=["Point", "Distance à μ1", "Distance à μ2", "Groupe"])

### Ce que vous devez voir
VOIR_TABLEAU_KM

C'est le tableau du corrigé. Mise à jour : le groupe 1 = {1}, donc μ₁ = 1 ; le groupe 2 = {3, 4, 9, 10, 14}, donc μ₂ = 40 / 5 = 8.

**Observez :** après cette première itération, le groupe 2 mélange encore des rôles très différents (3 et 14 répliques). Une seule itération suffit-elle ?

### Répéter jusqu'à ce que les groupes ne changent plus
La cellule répète les deux opérations. `[p for p in repliques if ...]` construit la liste des points qui vérifient la condition. On s'arrête quand les centres ne bougent plus : les groupes sont alors stables.

**Avant d'exécuter :** combien d'itérations faudra-t-il ? Quels seront les deux groupes finaux et leurs centres ?

In [ ]:
#@title Répéter affectation et mise à jour des centres
mu1 = mu1_depart
mu2 = mu2_depart
for iteration in range(1, 11):
    # 1. affecter chaque point au centre le plus proche
    groupe1 = [p for p in repliques if abs(p - mu1) <= abs(p - mu2)]
    groupe2 = [p for p in repliques if abs(p - mu1) > abs(p - mu2)]
    anciens_centres = (mu1, mu2)
    # 2. chaque centre devient la moyenne de son groupe (si le groupe n'est pas vide)
    if len(groupe1) > 0:
        mu1 = sum(groupe1) / len(groupe1)
    if len(groupe2) > 0:
        mu2 = sum(groupe2) / len(groupe2)
    print(f"Itération {iteration} : {groupe1} et {groupe2} → μ1 = {mu1:.2f}, μ2 = {mu2:.2f}")
    if (mu1, mu2) == anciens_centres:
        print("Les centres ne bougent plus : les groupes sont stables, arrêt.")
        break

### Ce que vous devez voir
```
VOIR_KM_BOUCLE
```
C'est le corrigé, bonus compris : à l'itération 2, 3 et 4 rejoignent le groupe 1 ; μ₁ = 8/3 ≈ 2,67 et μ₂ = 33/3 = 11. À l'itération 3, plus rien ne change. Résultat : {1, 3, 4}, les rôles secondaires, et {9, 10, 14}, les rôles principaux. C'est vous, et non l'algorithme, qui nommez ainsi les groupes.

**Modifiez :** partez de `mu1_depart` = 13 et `mu2_depart` = 14 (relancez les deux cellules) : obtenez-vous les mêmes groupes ? En combien d'itérations ?

### Vérifier avec KMeans de scikit-learn
`KMeans` fait le même travail. On peut lui imposer les centres de départ du TD (`init`), ou le laisser tirer des départs au hasard plusieurs fois (`n_init=10`) et garder la meilleure solution. L'**inertie** est la somme des carrés des distances entre chaque point et le centre de son groupe : plus elle est petite, plus les groupes sont serrés.

**Avant d'exécuter :** scikit-learn trouvera-t-il les mêmes groupes ? Calculez l'inertie : (1 − 8/3)² + (3 − 8/3)² + (4 − 8/3)² + (9 − 11)² + (10 − 11)² + (14 − 11)².

In [ ]:
#@title Vérifier avec KMeans de scikit-learn
from sklearn.cluster import KMeans

points = [[r] for r in repliques]               # scikit-learn attend une liste de lignes
depart = [[mu1_depart], [mu2_depart]]           # les centres de départ du TD
km = KMeans(n_clusters=2, init=depart, n_init=1).fit(points)
print("Départ du TD      → groupes :", km.labels_, "| centres :", km.cluster_centers_.ravel().round(2))
print("Inertie :", round(km.inertia_, 2))
km_hasard = KMeans(n_clusters=2, n_init=10, random_state=0).fit(points)   # 10 départs au hasard
print("Départs au hasard → groupes :", km_hasard.labels_, "| centres :", km_hasard.cluster_centers_.ravel().round(2))

### Ce que vous devez voir
```
VOIR_SKLEARN_KM
```
Les mêmes groupes et les mêmes centres, 2,67 et 11 ; l'inertie vaut 18,67. Avec des départs au hasard, les **numéros** de groupe sont inversés (0 et 1 échangés) : ces numéros sont arbitraires, seule compte la composition des groupes. Selon la version de scikit-learn, vous pouvez obtenir l'une ou l'autre numérotation.

### Combien de groupes ? La méthode du coude
L'inertie baisse toujours quand k augmente (avec 6 groupes pour 6 points, elle vaudrait 0). La **méthode du coude** retient la valeur de k à partir de laquelle elle ne baisse presque plus.

**Avant d'exécuter :** où sera le coude pour les six personnages ?

In [ ]:
#@title La méthode du coude : l'inertie selon le nombre de groupes k
inerties = []
for k in range(1, 5):
    km_k = KMeans(n_clusters=k, n_init=10, random_state=0).fit(points)
    inerties.append(km_k.inertia_)
    print(f"k = {k} : inertie = {km_k.inertia_:.2f}")
plt.figure(figsize=(5, 3))
plt.plot(range(1, 5), inerties, color="#2a78d6", marker="o", linewidth=2)
plt.xticks(range(1, 5))
plt.xlabel("Nombre de groupes k")
plt.ylabel("Inertie")
plt.title("Méthode du coude (répliques de six personnages)")
plt.grid(alpha=0.3)
plt.show()

### Ce que vous devez voir
```
VOIR_COUDE
```
et une courbe qui plonge de k = 1 à k = 2, puis s'aplatit : le coude est à k = 2, ce qui confirme le choix du TD.

**Observez :** avec k = 3, un groupe à part se forme-t-il pour le personnage principal (14 répliques) ? Le gain d'inertie justifie-t-il un troisième groupe ?

## À vous
1. Partie 1 : placez `livres_etudiant_5` sur 12 et relancez la partie 1. Notez a et b : un seul étudiant atypique change-t-il beaucoup la droite ?
2. Partie 2 : comparez `eta` = 0.1, 0.5, 1 et 1.1 avec 10 pas. Quelle valeur atteint le minimum le plus vite ? Laquelle ne l'atteint jamais ?
3. Partie 3 : partez de μ₁ = 13 et μ₂ = 14, puis de μ₁ = 9 et μ₂ = 10. Les groupes finaux dépendent-ils du point de départ, pour ces six personnages ?

## Pour le mini-projet : regrouper les réponses d'une enquête
Le sujet 2 du mini-projet applique les k-moyennes à une enquête. L'exemple ci-dessous est celui du cours : neuf réponses **fictives** à deux questions, les heures de lecture par semaine et les heures d'écrans par jour. Pour vos propres données, préparez un tableau CSV (dans Excel : « Enregistrer sous → CSV UTF-8 »), une ligne par répondant anonymisé et une colonne par question chiffrée.

- Dans Colab, cochez `charger_mon_fichier` : une fenêtre s'ouvre pour choisir le fichier.
- Dans Jupyter, placez le fichier à côté du carnet et écrivez son nom dans `nom_du_fichier`.

**Avant d'exécuter :** combien de lignes et de colonnes compte le tableau d'exemple ?

In [ ]:
#@title L'enquête du cours (données fictives)
import io

exemple_csv = """lecture_h_semaine,ecrans_h_jour
10,1
12,2
9,1
2,6
1,7
3,6
5,3
6,4
5,4"""
enquete = pd.read_csv(io.StringIO(exemple_csv))
print(enquete.shape)

### Ce que vous devez voir
`(9, 2)` : neuf répondants et deux colonnes, `lecture_h_semaine` et `ecrans_h_jour`.

In [ ]:
#@title À MODIFIER : charger votre propre fichier CSV (facultatif)
charger_mon_fichier = False  #@param {type:"boolean"}
nom_du_fichier = "mon_enquete.csv"  #@param {type:"string"}
separateur = ","  #@param [",", ";"]

if charger_mon_fichier:
    try:
        from google.colab import files        # ce module n'existe que dans Colab
        envoi = files.upload()                # une fenêtre s'ouvre : choisissez votre fichier
        nom_du_fichier = list(envoi)[0]
    except ImportError:
        print("Hors de Colab : le fichier", nom_du_fichier, "doit être à côté du carnet.")
    enquete = pd.read_csv(nom_du_fichier, sep=separateur, encoding="utf-8")
print(enquete.shape)
print(enquete.head())

### Ce que vous devez voir
Tant que la case n'est pas cochée, `(9, 2)` et les cinq premières réponses de l'exemple. Avec votre fichier, notez le nom exact de deux colonnes chiffrées : vous les taperez dans la cellule suivante.

**Avant d'exécuter la cellule suivante :** où sera le coude ? Quels profils de lecteurs attendez-vous ?

In [ ]:
#@title À MODIFIER : choisir deux colonnes et le nombre de groupes k
colonne_x = "lecture_h_semaine"  #@param {type:"string"}
colonne_y = "ecrans_h_jour"  #@param {type:"string"}
k = 3  #@param {type:"slider", min:2, max:5, step:1}

donnees = enquete[[colonne_x, colonne_y]]
for nombre in range(1, 6):                     # la méthode du coude
    inertie = KMeans(n_clusters=nombre, n_init=10, random_state=0).fit(donnees).inertia_
    print(f"k = {nombre} : inertie = {inertie:.1f}")
km_enquete = KMeans(n_clusters=k, n_init=10, random_state=0).fit(donnees)
print("Groupes :", km_enquete.labels_)
print("Centres :", km_enquete.cluster_centers_.round(1).tolist())

### Ce que vous devez voir
```
VOIR_ENQUETE
```
Ce sont les valeurs du cours. Le coude se situe à k = 3. Les numéros de groupe sont arbitraires (ils peuvent différer selon la version de scikit-learn) : c'est à vous d'interpréter chaque centre, par exemple [10,3 ; 1,3] pour les grands lecteurs, [2,0 ; 6,3] pour les grands utilisateurs d'écrans et [5,3 ; 3,7] pour les profils intermédiaires.

**Avant d'exécuter la cellule suivante :** dessinez à la main le nuage de points et entourez les trois groupes.

In [ ]:
#@title Dessiner les groupes : une couleur et une forme par groupe
couleurs = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
formes = ["o", "s", "^", "D", "v"]
plt.figure(figsize=(6, 4))
for g in range(k):
    membres = donnees[km_enquete.labels_ == g]       # les répondants du groupe g
    plt.scatter(membres[colonne_x], membres[colonne_y], color=couleurs[g], marker=formes[g], s=70, label=f"Groupe {g}")
centres = km_enquete.cluster_centers_
plt.scatter(centres[:, 0], centres[:, 1], color="black", marker="x", s=100, label="Centres")
plt.xlabel(colonne_x)
plt.ylabel(colonne_y)
plt.title(f"k-moyennes sur l'enquête, k = {k}")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### Ce que vous devez voir
Trois nuages bien séparés de trois points chacun, avec une croix noire au centre de chacun : les grands lecteurs en bas à droite, les grands utilisateurs d'écrans en haut à gauche, les profils intermédiaires au milieu.

**Modifiez :** placez `k` sur 2, puis sur 4, et relancez les deux cellules. Quels groupes fusionnent, lesquels se scindent ? Pour votre mini-projet, si vos colonnes ont des échelles très différentes (des heures et des notes sur 20, par exemple), pensez à les standardiser avant le regroupement : le carnet `02-questionnaire` le montre.

## À rendre
Répondez par écrit, en quelques lignes chacune :
1. Interprétez la droite ŷ = 0,9 x + 1,3 de l'exercice 2 a : que signifient a et b pour ces étudiants fictifs ? Pourquoi la prédiction pour 12 heures par semaine est-elle douteuse ?
2. Décrivez ce que vous avez observé avec η = 0,1, η = 1 et η = 1,1 dans la descente de gradient. En quoi peut-on dire qu'un modèle « apprend » quand il descend la pente de son erreur ?
3. Les k-moyennes ont formé les groupes {1, 3, 4} et {9, 10, 14}. Qui décide que ce sont des « rôles secondaires » et des « rôles principaux » : l'algorithme ou le chercheur ? Proposez, dans votre spécialité, une question de recherche qu'un regroupement pourrait éclairer.

## Ce qu'il faut retenir
- La régression linéaire prédit un nombre avec la droite des moindres carrés, a = Σ(x − x̄)(y − ȳ) / Σ(x − x̄)² et b = ȳ − a·x̄ ; elle ne vaut que dans l'intervalle observé, et une corrélation n'est pas une cause.
- La descente de gradient ajuste les paramètres pas à pas, w ← w − η · J′(w) : c'est ainsi qu'un modèle apprend. Un pas trop grand diverge, un pas trop petit est lent.
- Les k-moyennes forment des groupes sans étiquettes en alternant affectation et recalcul des centres ; le résultat dépend du départ et de k (méthode du coude), et c'est au chercheur de l'interpréter.

**Dans le cours :** [Quelques algorithmes d'apprentissage automatique](https://progremer04.github.io/canva-au-for-ang-/#c3-algorithmes) · TD du chapitre 3 : [exercice 2 a](https://progremer04.github.io/canva-au-for-ang-/#td-ch3-ex2a), [exercice 2 b](https://progremer04.github.io/canva-au-for-ang-/#td-ch3-ex2b), [exercice 2 d](https://progremer04.github.io/canva-au-for-ang-/#td-ch3-ex2d)